# Router fine-tuning (Qwen2.5-1.5B-Instruct + LoRA) on a free Colab T4
Runtime > Change runtime type > T4 GPU. Replace REPO_URL with your GitHub repo.

In [ ]:
REPO_URL = 'https://github.com/<you>/cadral-ai-ops-agent'
!git clone $REPO_URL repo && cd repo && pip install -q transformers peft trl datasets accelerate httpx numpy pypdf
%cd repo

In [ ]:
!python finetune/make_dataset.py
# 1) rules baseline, 2) BASE small model (zero-shot)
!python finetune/evaluate.py --heuristic
!python finetune/evaluate.py --hf Qwen/Qwen2.5-1.5B-Instruct --gpu-usd-per-hour 0.35

In [ ]:
# 3) fine-tune (about 10 minutes on a T4)
!python finetune/train_lora.py --base Qwen/Qwen2.5-1.5B-Instruct --out finetune/out/router-lora

In [ ]:
# 4) FINE-TUNED model on the same held-out test set
!python finetune/evaluate.py --hf Qwen/Qwen2.5-1.5B-Instruct --lora finetune/out/router-lora --gpu-usd-per-hour 0.35

In [ ]:
# 5) large API model for comparison (needs a free Groq key)
import os; os.environ['GROQ_API_KEY'] = '<paste key>'
!python finetune/evaluate.py --groq openai/gpt-oss-120b

In [ ]:
import json, pandas as pd
pd.DataFrame(json.load(open('finetune/results.json')))[['model','accuracy','macro_f1','latency_ms_mean','latency_ms_p95','cost_usd_per_1k_requests']]
# Download finetune/results.json and commit it. Optional: merge the adapter and serve it (llama.cpp / Ollama), then set FT_BASE_URL in .env.